# DTP-Fact: one-click Kaggle runner

Attach `dtp_fact_kaggle_upload_bundle_FINAL_2026-10-25.zip` as a **private** Kaggle dataset, enable a GPU and Internet, then select **Run all**. The notebook automatically finds and extracts the packages, installs the project, runs its tests and synthetic diagnostic, and summarizes the real candidate snapshot.

The included ClaimReview records are screening candidates, not audited training data. A scientific CUDA run starts only when a separate `dtp_fact_final_data*.zip` with the audited layout is attached. Synthetic diagnostic values must never be reported as paper results.

In [1]:
# `auto` runs the final study only when an audited final-data ZIP is attached.
# Use `diagnostic` to force the quick software checks only.
RUN_MODE = "auto"
assert RUN_MODE in {"auto", "diagnostic", "final"}

In [2]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path.cwd()
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "_local_notebook_work"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def first_match(root, pattern):
    matches = sorted(root.rglob(pattern))
    return matches[0] if matches else None

asset_root = SEARCH_ROOT
master_zip = first_match(SEARCH_ROOT, "dtp_fact_kaggle_upload_bundle*.zip")
if master_zip is not None:
    asset_root = WORK_ROOT / "dtp_fact_upload_assets"
    if asset_root.exists():
        shutil.rmtree(asset_root)
    asset_root.mkdir(parents=True)
    with zipfile.ZipFile(master_zip) as archive:
        archive.extractall(asset_root)
    print(f"Extracted upload bundle: {master_zip}")

code_zip = first_match(asset_root, "dtp_fact_code*.zip") or first_match(SEARCH_ROOT, "dtp_fact_code*.zip")
candidate_zip = first_match(asset_root, "dtp_fact_candidate_snapshot*.zip") or first_match(SEARCH_ROOT, "dtp_fact_candidate_snapshot*.zip")
if code_zip is None:
    raise FileNotFoundError("Could not find dtp_fact_code*.zip in the attached Kaggle dataset")

code_extract = WORK_ROOT / "dtp_fact_code"
if code_extract.exists():
    shutil.rmtree(code_extract)
code_extract.mkdir(parents=True)
with zipfile.ZipFile(code_zip) as archive:
    archive.extractall(code_extract)
project_root = first_match(code_extract, "ET2/experiments/pyproject.toml")
if project_root is None:
    raise FileNotFoundError("The code ZIP does not contain ET2/experiments/pyproject.toml")
PROJECT_ROOT = project_root.parents[1]
EXPERIMENT_ROOT = PROJECT_ROOT / "experiments"

CANDIDATE_ROOT = None
if candidate_zip is not None:
    candidate_extract = WORK_ROOT / "dtp_fact_candidate_snapshot"
    if candidate_extract.exists():
        shutil.rmtree(candidate_extract)
    candidate_extract.mkdir(parents=True)
    with zipfile.ZipFile(candidate_zip) as archive:
        archive.extractall(candidate_extract)
    queue = first_match(candidate_extract, "screening_queue.csv")
    CANDIDATE_ROOT = queue.parent if queue else None

print(f"Project: {PROJECT_ROOT}")
print(f"Candidate snapshot: {CANDIDATE_ROOT or 'not attached'}")

Project: D:\1.msc\ET2\kaggle\verify-final-input\_local_notebook_work\dtp_fact_code\ET2
Candidate snapshot: D:\1.msc\ET2\kaggle\verify-final-input\_local_notebook_work\dtp_fact_candidate_snapshot\us_tpmfc_candidate_snapshot


In [3]:
def run(command, cwd=EXPERIMENT_ROOT):
    print("+", " ".join(str(part) for part in command))
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[analysis,test,retrieval,ml]"])
os.environ["PYTHONPATH"] = str(EXPERIMENT_ROOT / "src")
os.environ["HF_HOME"] = str(WORK_ROOT / "hf-cache")
print("Installation complete")

+ D:\1.msc\ET2\experiments\.venv\Scripts\python.exe -m pip install -q -e .[analysis,test,retrieval,ml]


Installation complete


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

run([sys.executable, "-m", "pytest", "-q"])
run([sys.executable, "scripts/run_smoke_test.py", "--config", "configs/base.yaml"])
smoke_metrics = json.loads((EXPERIMENT_ROOT / "outputs/smoke/metrics.json").read_text(encoding="utf-8"))
print("Software diagnostic status:", smoke_metrics["status"])

CUDA available: False
+ D:\1.msc\ET2\experiments\.venv\Scripts\python.exe -m pytest -q


+ D:\1.msc\ET2\experiments\.venv\Scripts\python.exe scripts/run_smoke_test.py --config configs/base.yaml


Software diagnostic status: synthetic_software_diagnostic_only


In [5]:
import pandas as pd
from IPython.display import display

if CANDIDATE_ROOT is None:
    print("Candidate snapshot was not found; software checks still completed.")
else:
    collection = json.loads((CANDIDATE_ROOT / "collection_manifest.json").read_text(encoding="utf-8"))
    queue = pd.read_csv(CANDIDATE_ROOT / "screening_queue.csv")
    print("Deduplicated ClaimReview candidates:", collection["candidate_count_after_deduplication"])
    print("Screening queue rows:", len(queue))
    display(pd.crosstab(queue["suggested_label"], queue["temporal_split"], margins=True))
    display(queue[["screening_rank", "publisher", "claim_text", "textual_rating", "suggested_label", "review_url"]].head(20))
    print("These rows are screening candidates only; they are not gold labels or paper results.")

Deduplicated ClaimReview candidates: 32898
Screening queue rows: 1200


temporal_split,test,train,validation,All
suggested_label,,,,
insufficient_evidence,75,252,73,400
refuted,120,200,80,400
supported,120,200,80,400
All,315,652,233,1200


,screening_rank,publisher,claim_text,textual_rating,suggested_label,review_url
0,1,PolitiFact,"Image claims to show a list of ""Republican mem...",True,supported,https://politifact.com/factchecks/2022/aug/30/...
1,2,Snopes,A viral photograph represents former U.S. Pres...,True,supported,https://www.snopes.com/fact-check/trump-fulton...
2,3,PolitiFact,"""As we see unemployment increase to Depression...",Accurate,supported,https://politifact.com/article/2020/may/14/fac...
3,4,PolitiFact,Says that under his administration there was t...,True,supported,https://politifact.com/article/2020/jan/23/fac...
4,5,Snopes,"A video from a live interview on June 29, 2023...",True,supported,https://www.snopes.com/fact-check/biden-walks-...
5,6,Snopes,In a video displayed during U.S. President Don...,True,supported,https://www.snopes.com/fact-check/trump-peacef...
6,7,Snopes,"On Dec. 8, 2020, U.S. President Donald Trump r...",True,supported,https://www.snopes.com/fact-check/amy-coney-ba...
7,8,Snopes,U.S. President Donald Trump has suggested he w...,True,supported,https://www.snopes.com/fact-check/trump-on-mou...
8,9,Snopes,A photograph shows U.S. President Donald Trump...,True,supported,https://www.snopes.com/fact-check/trump-bloomb...
9,10,Snopes,An AP-distributed photograph shows Senate Majo...,True,supported,https://www.snopes.com/fact-check/mitch-mcconn...


These rows are screening candidates only; they are not gold labels or paper results.


In [6]:
final_zip = first_match(SEARCH_ROOT, "dtp_fact_final_data*.zip")
should_run_final = RUN_MODE == "final" or (RUN_MODE == "auto" and final_zip is not None)

if not should_run_final:
    print("FINAL RUN SKIPPED: no audited dtp_fact_final_data*.zip is attached.")
    print("The notebook completed the code tests, smoke diagnostic and candidate-snapshot check.")
else:
    if final_zip is None:
        raise FileNotFoundError("RUN_MODE='final' requires an attached dtp_fact_final_data*.zip")
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU before the scientific run")
    final_extract = WORK_ROOT / "dtp_fact_final_data"
    if final_extract.exists():
        shutil.rmtree(final_extract)
    final_extract.mkdir(parents=True)
    with zipfile.ZipFile(final_zip) as archive:
        archive.extractall(final_extract)
    dataset_manifest = first_match(final_extract, "frozen/dataset_manifest.json")
    if dataset_manifest is None:
        raise FileNotFoundError("Final-data ZIP is missing frozen/dataset_manifest.json")
    FINAL_ROOT = dataset_manifest.parents[1]
    required = [
        "collection_manifest.json", "conflict_donors.jsonl", "image_manifest.csv",
        "pilot_claims_with_evidence.jsonl", "pilot_evidence.jsonl", "pilot_assessment.json",
        "frozen/claims.jsonl", "frozen/evidence.jsonl", "frozen/dataset_manifest.json",
    ]
    missing = [name for name in required if not (FINAL_ROOT / name).exists()]
    if missing:
        raise FileNotFoundError(f"Final-data ZIP is incomplete: {missing}")

    processed = EXPERIMENT_ROOT / "data/processed"
    pilot_outputs = EXPERIMENT_ROOT / "outputs/pilot"
    processed.mkdir(parents=True, exist_ok=True)
    pilot_outputs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(FINAL_ROOT / "pilot_claims_with_evidence.jsonl", processed / "pilot_claims_with_evidence.jsonl")
    shutil.copy2(FINAL_ROOT / "pilot_evidence.jsonl", processed / "pilot_evidence.jsonl")
    shutil.copy2(FINAL_ROOT / "pilot_assessment.json", pilot_outputs / "pilot_assessment.json")

    run([
        sys.executable, "scripts/preflight_full_run.py",
        "--collection-manifest", FINAL_ROOT / "collection_manifest.json",
        "--conflict-manifest", FINAL_ROOT / "conflict_donors.jsonl",
    ])

    command = [
        sys.executable, "scripts/run_full_study.py",
        "--config", "configs/base.yaml",
        "--frozen-data-directory", FINAL_ROOT / "frozen",
        "--claim-image-manifest", FINAL_ROOT / "image_manifest.csv",
        "--conflict-manifest", FINAL_ROOT / "conflict_donors.jsonl",
        "--output-directory", WORK_ROOT / "final_run",
        "--paper-table-directory", WORK_ROOT / "generated_tables",
        "--device", "cuda", "--resume",
    ]
    evidence_images = FINAL_ROOT / "evidence_image_manifest.csv"
    evidence_text = FINAL_ROOT / "evidence_text.jsonl"
    if evidence_images.exists():
        command.extend(["--evidence-image-manifest", evidence_images])
    if evidence_text.exists():
        command.extend(["--evidence-text", evidence_text])
    run(command)
    print("FINAL RUN COMPLETE")
    print("Preserve:", WORK_ROOT / "final_run")
    print("Generated paper tables:", WORK_ROOT / "generated_tables")

FINAL RUN SKIPPED: no audited dtp_fact_final_data*.zip is attached.
The notebook completed the code tests, smoke diagnostic and candidate-snapshot check.
